In [28]:
import pandas as pd 
import numpy as np
import json
from difflib import SequenceMatcher
import re
import yaml
from utils import normalize_text, similarity, extract_numeric_candidates, is_near_keyword

In [29]:
df = pd.read_csv("0130_Ledger.csv", dtype=str)

In [30]:
with open ("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)
print(cfg)

{'anchors': {'name': ['PAID TO3', 'PAY TO', 'FAVOUR OF', 'TO', 'ISSUED TO', 'NAME', 'CUST NAME', 'CHEQUE:#', 'CUSTOMER NAME', 'CUS NAME'], 'cheque': ['CHEQUE', 'CHQ', 'CHEQ', 'CHQNO', 'CHE NO', 'cheque']}, 'thresholds': {'name_similarity': 0.85, 'min_confidence': 0.6}}


In [31]:
df['particulars'] = (
    df['particulars']
    .str.strip()
    .str.upper()
    .str.replace(r"/s{2,}", " ")
    .str.replace(r'[,:;/]', ' ')
)
df['particulars'].head(10)

0    CHQ NO 772879 ISSUED TO SPECTRONIC NETWORK ( A...
1    CHQ NO 772880 ISSUED TO ADLB SOLUTIONS ( ADV.N...
2    RTGS FROM HO TREASURY FUND TRANSFER, HDFC BANK...
3    RTGS FROM HO TREASURY FUND TRANSFER, HDFC BANK...
4    CHQ NO 772881 ISSUED TO PRABHU MONEY TRANSFER ...
5    CHQ NO 772882 ISSUED TO IME INDIA PRIVATE LIMI...
6    CHQ NO 772883 ISSUED TO MUTHOOT FOREX LTD ( AD...
7    RTGS FROM HO TREASURY FUND TRANSFER, HDFC BANK...
8    RTGS FROM HO TREASURY FUND TRANSFER, HDFC BANK...
9    RTGS FROM HO TREASURY FUND TRANSFER, HDFC BANK...
Name: particulars, dtype: str

In [32]:
df['extracted_name'] = df['particulars'].str.extract(r'(/b/d{5,15}/b)')
df.tail(10)

,txn._date,cheque_no.,particulars,credit_amount,debit_amount,source,extracted_name
144,2025-11-28,772951,CH NO 772951 ISSUED FOR PRABHU MONEY ( ADV.NO....,10000000.0,0.0,0130_Ledger,NaN
145,2025-11-28,772952,CH NO 772952 ISSUED FOR IME ( ADV.NO. 4305 FRO...,10000000.0,0.0,0130_Ledger,NaN
146,2025-11-28,772953,CH NO 772953 ISSUED FOR MUTHOOT FOREX ( ADV.NO...,2000000.0,0.0,0130_Ledger,NaN
147,2025-11-28,NaN,"RTGS FROM HO TREASURY FUND TRANSFER, HDFC BANK...",0.0,10000000.0,0130_Ledger,NaN
148,2025-11-28,NaN,"RTGS FROM HO TREASURY FUND TRANSFER, HDFC BANK...",0.0,10000000.0,0130_Ledger,NaN
149,2025-11-28,NaN,"RTGS FROM HO TREASURY FUND TRANSFER, HDFC BANK...",0.0,2000000.0,0130_Ledger,NaN
150,2025-11-28,772954,CH NO 772954 ISSUED FOR ADLB SOLUTION ( ADV.NO...,378888.0,0.0,0130_Ledger,NaN
151,2025-11-28,772955,CH NO 772955 ISSUED FOR SPECTRONIC NETWORK ( A...,3633513.0,0.0,0130_Ledger,NaN
152,2025-11-28,NaN,"RTGS FROM HO TREASURY FUND TRANSFER, HDFC BANK...",0.0,378888.0,0130_Ledger,NaN
153,2025-11-28,NaN,"RTGS FROM HO TREASURY FUND TRANSFER, HDFC BANK...",0.0,3633513.0,0130_Ledger,NaN


In [33]:
cust_name = json.load(open("customer_names.json", "r"))
cust_name

['a dhivya',
 'a prabhu kumar',
 'abishek a g',
 'abuthahir',
 'adlb solutions',
 'allam prabhu kumar',
 'ambati nagaraj bharath',
 'ar gold',
 'arumugam karuppannan',
 'avanigadda sesha ratnam',
 'avanigadda sesharatnam',
 'b manohar babu',
 'b prameela',
 'b ramesh',
 'b srinivas',
 'balakrishnan',
 'boddula pavan',
 'busanaboyina sujatha',
 'c venkateswara rao',
 'chandru r',
 'chandyala venkateswara rao',
 'chinnapillai p',
 'dadi sampath',
 'divya a',
 'e ashok',
 'eerla ashok kumar',
 'elavarasan a',
 'elavarasu',
 'g chandra shekar',
 'g hemalatha',
 'g sunny chowdary',
 'gadapa hemalatha',
 'gadila ajay kumar',
 'gorripati sunny chowdhary',
 'husainsab imamsab hadignal',
 'ime india india pvt ltd',
 'ime india pvt ltd',
 'j krishna pradeep',
 'janne srivani',
 'k sai ram charan',
 'k santhosh kumar',
 'k venkatesh',
 'kade santhosh',
 'karnan',
 'karnan s',
 'karthick k',
 'karupasamy',
 'karuppusamy',
 'katuri sriram charan',
 'kavi priya',
 'koppala mary kamala',
 'korlapu re

In [46]:
LEGAL_MAP = {
    "PRIVATE": "PVT",
    "PVT.": "PVT",
    "PVT": "PVT",

    "LIMITED": "LTD",
    "LTD.": "LTD",
    "LTD": "LTD",

    "COMPANY": "CO",
    "CO.": "CO",

    "ENTERPRISES": "ENT",
    "ENTERPRISE": "ENT",

    "INDUSTRIES": "IND",
    "INDUSTRY": "IND"
}


In [48]:
def load_names(path):
    with open(path, "r", encoding="utf-8") as f:
        name = [n.upper().strip() for n in json.load(f)]
        return name
    
def normalize(text):
    if not text:
        return ""
    text = text.upper()
    text = re.sub(r'[,:;/]', ' ', text)
    text = re.sub(r'[^A-Z0-9 ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def normalize_entity_name(text):
    if not text:
        return ""

    text = normalize(text)   
    tokens = []

    for t in text.split():
        tokens.append(LEGAL_MAP.get(t, t))

    return " ".join(tokens)



def extract_anchor_window(text, anchors, window=10):
    words = text.split()
    anchors = set(anchors)

    for i, w in enumerate(words):
        if w in anchors:
            start = i + 1
            end = min(len(words), start + window)
            return " ".join(words[start:end])

    return "".join(words[-window:])



def similarity(a, b):
    a_set = set(a.split())
    b_set = set(b.split())
    if not a_set or not b_set:
        return 0.0
    # return len(a_set & b_set) / max(len(a_set), len(b_set))
    
    return len(a_set & b_set) / len(b_set)


def match_name(text_window, name_list, threshold):
    if not text_window:
        return None, 0.0

    best_match, best_score = None, 0.0

    name_list = [normalize_entity_name(n) for n in name_list]

    for name in name_list:
        if text_window == name:
            return name, 1.0

        score = similarity(text_window, name)
        if score > best_score:
            best_match, best_score = name, score

    if best_score >= threshold:
        return best_match, round(best_score, 2)

    return None, 0.0





def process_row(raw_text, cfg, name_list):
    text = normalize(raw_text)

    # Name extraction
    name_window = extract_anchor_window(
        text, cfg["anchors"]["name"]
    )
    name_window = normalize_entity_name(name_window)
    name, name_conf = match_name(
        name_window, name_list, cfg["thresholds"]["name_similarity"]
    )

    # Cheque extraction
    # cheque, cheque_conf = extract_cheque(
    #     text, cfg["anchors"]["cheque"]
    # )

    reason = None

    if not name:
        status = "REVIEW"
        reason = "NAME_NOT_FOUND"

    elif name_conf < cfg["thresholds"]["min_confidence"]:
        status = "REVIEW"
        reason = "LOW_CONFIDENCE"

    else:
        status = "OK"

    return {
        "extracted_name": name,
        "name_confidence": name_conf,
        # "cheque_no": cheque,
        # "cheque_confidence": cheque_conf,
        "row_status": status,
        "review_reason": reason
    }


def run_pipeline(df, text_col, cfg, name_list):
    results = df[text_col].apply(
        lambda x: process_row(x, cfg, name_list)
    )

    return pd.concat(
        [df, results.apply(pd.Series)],
        axis=1
    )



In [49]:
result_df = run_pipeline(
    df = df,
    text_col = "particulars",
    cfg = cfg,
    name_list = load_names("customer_names.json")
)
result_df
# result_df

,txn._date,cheque_no.,particulars,credit_amount,debit_amount,source,extracted_name,extracted_name,name_confidence,row_status,review_reason
0,2025-11-01,772879,CHQ NO 772879 ISSUED TO SPECTRONIC NETWORK ( A...,7195803.0,0.0,0130_Ledger,NaN,SPECTRONIC NETWORK,1.0,OK,NaN
1,2025-11-01,772880,CHQ NO 772880 ISSUED TO ADLB SOLUTIONS ( ADV.N...,863792.0,0.0,0130_Ledger,NaN,ADLB SOLUTIONS,1.0,OK,NaN
2,2025-11-01,NaN,"RTGS FROM HO TREASURY FUND TRANSFER, HDFC BANK...",0.0,863792.0,0130_Ledger,NaN,NaN,0.0,REVIEW,NAME_NOT_FOUND
3,2025-11-01,NaN,"RTGS FROM HO TREASURY FUND TRANSFER, HDFC BANK...",0.0,7195803.0,0130_Ledger,NaN,NaN,0.0,REVIEW,NAME_NOT_FOUND
4,2025-11-03,772881,CHQ NO 772881 ISSUED TO PRABHU MONEY TRANSFER ...,20000000.0,0.0,0130_Ledger,NaN,PRABHU MONEY TRANSFER PVT LTD,1.0,OK,NaN
...,...,...,...,...,...,...,...,...,...,...,...
149,2025-11-28,NaN,"RTGS FROM HO TREASURY FUND TRANSFER, HDFC BANK...",0.0,2000000.0,0130_Ledger,NaN,NaN,0.0,REVIEW,NAME_NOT_FOUND
150,2025-11-28,772954,CH NO 772954 ISSUED FOR ADLB SOLUTION ( ADV.NO...,378888.0,0.0,0130_Ledger,NaN,NaN,0.0,REVIEW,NAME_NOT_FOUND
151,2025-11-28,772955,CH NO 772955 ISSUED FOR SPECTRONIC NETWORK ( A...,3633513.0,0.0,0130_Ledger,NaN,NaN,0.0,REVIEW,NAME_NOT_FOUND
152,2025-11-28,NaN,"RTGS FROM HO TREASURY FUND TRANSFER, HDFC BANK...",0.0,378888.0,0130_Ledger,NaN,NaN,0.0,REVIEW,NAME_NOT_FOUND


In [50]:
result_df.to_csv("output.csv", index=False)